In [ ]:
import numpy as np
import pandas as ps
#import matplotlib.pyplot as plt
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
#from sklearn.preprocessing import Imputer
from sklearn.impute import SimpleImputer
import sklearn.metrics as skm

from itertools import combinations 
from functools import reduce

from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

get_ipython().magic('matplotlib inline') 

In [ ]:
dta = ps.read_csv("Downloads/E0.csv")

In [ ]:
def ada_model_py2(filename, gw = 0):
    dta = ps.read_csv(filename)
    max_gw = dta.Round.max()
    gameweek =  max_gw if gw == 0 else gw
    
    feature_columns = ['B365H', 'B365D', 'B365A', 'BWH', 'BWD', 'BWA', 'IWH',
                    'IWD', 'IWA','LBH', 'LBD', 'LBA', 'PSH', 'PSD', 'PSA',
                    'SOH', 'SOD', 'SOA', 'SBH', 'SBD', 'SBA', 'SJH', 'SJD',
                    'SJA', 'SYH', 'SYD','SYA', 'VCH', 'VCD', 'VCA', 'WHH',
                    'WHD', 'WHA']
                    #,'HS','AS','HST','AST','HF','AF','HC','AC','HY','AY','HR','AR']

    historic_final_columns = ['HomeTeam', 'AwayTeam', 'Round','FTR','B365H', 'B365D', 'B365A']
    
    
    #Find the row numbers that should be used for training and testing.
    train_idx = np.array(dta.Round < gameweek)
    test_idx = np.array(dta.Round == gameweek)


    #Arrays where the match results are stored in
    results_train = np.array(dta.FTR[train_idx])
    results_test = np.array(dta.FTR[test_idx])

    #Column numbers for odds for the three outcomes 
    cidx_home = [i for i, col in enumerate(dta.columns) if col[-1] in 'H' and col in feature_columns]
    cidx_draw = [i for i, col in enumerate(dta.columns) if col[-1] in 'D' and col in feature_columns]
    cidx_away = [i for i, col in enumerate(dta.columns) if col[-1] in 'A' and col in feature_columns]
 
    #The three feature matrices for training
    feature_train_home = dta.ix[train_idx, cidx_home].as_matrix()
    feature_train_draw = dta.ix[train_idx, cidx_draw].as_matrix()
    feature_train_away = dta.ix[train_idx, cidx_away].as_matrix()
 
    #The three feature matrices for testing
    feature_test_home = dta.ix[test_idx, cidx_home].as_matrix()
    feature_test_draw = dta.ix[test_idx, cidx_draw].as_matrix()
    feature_test_away = dta.ix[test_idx, cidx_away].as_matrix()
 
    train_arrays = [feature_train_home, feature_train_draw, feature_train_away]
                                     
    test_arrays = [feature_test_home, feature_test_draw, feature_test_away]
 
    imputed_training_matrices = []
    imputed_test_matrices = []
 
    for idx, farray in enumerate(train_arrays):
        imp = Imputer(strategy='mean', axis=1) #0: column, 1:rows
        farray = imp.fit_transform(farray)
        test_arrays[idx] = imp.fit_transform(test_arrays[idx])
     
        imputed_training_matrices.append(farray)
        imputed_test_matrices.append(test_arrays[idx])
 
    #merge the imputed arrays
    feature_train = np.concatenate(imputed_training_matrices, axis=1)
    feature_test = np.concatenate(imputed_test_matrices, axis=1)

    adb = AdaBoostClassifier(DecisionTreeClassifier(max_depth=3),n_estimators=50000,learning_rate=0.2, random_state=42)

    adb = adb.fit(feature_train, results_train)
 
    training_pred = adb.predict(feature_train)
    print (skm.confusion_matrix(list(training_pred), list(results_train)))

    test_pred = adb.predict(feature_test)

    gameweek_predictions = dta[dta['Round']== gameweek][historic_final_columns]
    gameweek_predictions['PredictedResult'] = test_pred
       
    return gameweek_predictions

In [ ]:
def ada_model(filename, gw=0):
    dta = ps.read_csv(filename)
    max_gw = dta.Round.max()
    gameweek = max_gw if gw == 0 else gw

    feature_columns = [
        'B365H', 'B365D', 'B365A', 'BWH', 'BWD', 'BWA', 'IWH',
        'IWD', 'IWA', 'LBH', 'LBD', 'LBA', 'PSH', 'PSD', 'PSA',
        'SOH', 'SOD', 'SOA', 'SBH', 'SBD', 'SBA', 'SJH', 'SJD',
        'SJA', 'SYH', 'SYD', 'SYA', 'VCH', 'VCD', 'VCA',
        'WHH', 'WHD', 'WHA'
    ]

    historic_final_columns = [
        'HomeTeam', 'AwayTeam', 'Round', 'FTR', 'B365H', 'B365D', 'B365A'
    ]

    # Boolean masks for train/test
    train_idx = dta.Round < gameweek
    test_idx = dta.Round == gameweek

    # Match results
    results_train = dta.loc[train_idx, "FTR"].to_numpy()
    results_test = dta.loc[test_idx, "FTR"].to_numpy()

    # Column indices for odds
    cidx_home = [i for i, col in enumerate(dta.columns) if col.endswith("H") and col in feature_columns]
    cidx_draw = [i for i, col in enumerate(dta.columns) if col.endswith("D") and col in feature_columns]
    cidx_away = [i for i, col in enumerate(dta.columns) if col.endswith("A") and col in feature_columns]

    # Feature matrices for training
    feature_train_home = dta.loc[train_idx].iloc[:, cidx_home].to_numpy()
    feature_train_draw = dta.loc[train_idx].iloc[:, cidx_draw].to_numpy()
    feature_train_away = dta.loc[train_idx].iloc[:, cidx_away].to_numpy()

    # Feature matrices for testing
    feature_test_home = dta.loc[test_idx].iloc[:, cidx_home].to_numpy()
    feature_test_draw = dta.loc[test_idx].iloc[:, cidx_draw].to_numpy()
    feature_test_away = dta.loc[test_idx].iloc[:, cidx_away].to_numpy()

    train_arrays = [feature_train_home, feature_train_draw, feature_train_away]
    test_arrays = [feature_test_home, feature_test_draw, feature_test_away]

    imputed_training_matrices = []
    imputed_test_matrices = []

    for idx, farray in enumerate(train_arrays):
        # SimpleImputer works column-wise (axis=0)
        imp = SimpleImputer(strategy="mean")
        farray = imp.fit_transform(farray)
        test_arrays[idx] = imp.transform(test_arrays[idx])

        imputed_training_matrices.append(farray)
        imputed_test_matrices.append(test_arrays[idx])

    # Merge the imputed arrays
    feature_train = np.concatenate(imputed_training_matrices, axis=1)
    feature_test = np.concatenate(imputed_test_matrices, axis=1)

    # Train AdaBoost
    adb = AdaBoostClassifier(
        DecisionTreeClassifier(max_depth=3),
        n_estimators=50000,
        learning_rate=0.2,
        random_state=42
    )
    adb = adb.fit(feature_train, results_train)

    training_pred = adb.predict(feature_train)
    print(skm.confusion_matrix(results_train, training_pred))

    test_pred = adb.predict(feature_test)

    gameweek_predictions = dta.loc[dta['Round'] == gameweek, historic_final_columns].copy()
    gameweek_predictions['PredictedResult'] = test_pred

    return gameweek_predictions

In [ ]:
models = [ada_model]

global total_fixtures
global total_predicted

total_fixtures = 0
total_predicted = 0

def eval_predictions(gameweek_predictions):
  
    no_predicted = 0
    if gameweek_predictions['FTR'].isnull().all():
        print ("no full time results")
        no_predicted = 0
    else:
        print ("evaluate results")                       
        for index, row in gameweek_predictions.iterrows():
            if row['FTR'] == row['PredictedResult']:
                no_predicted += 1            
          
        total_fixtures  += len(gameweek_predictions)
        total_predicted += no_predicted
        
    return no_predicted / len(gameweek_predictions)

def evaluate_latest_gw_run():    
    for m in models:
        weekly_score = []
        for i in range(2,5):
            predictions = m("Downloads/E0.csv",i)
            #print (predictions)
            #predictions['TakenOdds'] = collect_predicted_odds(predictions)

            n = eval_predictions(predictions)
            print (n)
            # not always 10 events so normalized to no_correct / len(predictions)
            weekly_score.append(n)
        print(weekly_score)
        histogram_plot_score(weekly_score)
        z_score = sum(weekly_score)/len(weekly_score)
        print(z_score)
        
    return predictions

a = evaluate_latest_gw_run()

In [4]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
import sklearn.metrics as skm


def ada_model(filename, gw=0):
    dta = pd.read_csv(filename)
    max_gw = dta.Round.max()
    gameweek = max_gw if gw == 0 else gw

    feature_columns = [
        'B365H', 'B365D', 'B365A', 'BWH', 'BWD', 'BWA', 'IWH',
        'IWD', 'IWA', 'LBH', 'LBD', 'LBA', 'PSH', 'PSD', 'PSA',
        'SOH', 'SOD', 'SOA', 'SBH', 'SBD', 'SBA', 'SJH', 'SJD',
        'SJA', 'SYH', 'SYD', 'SYA', 'VCH', 'VCD', 'VCA',
        'WHH', 'WHD', 'WHA'
    ]

    historic_final_columns = [
        'HomeTeam', 'AwayTeam', 'Round', 'FTR', 'B365H', 'B365D', 'B365A'
    ]

    # Boolean masks for train/test
    train_idx = dta.Round < gameweek
    test_idx = dta.Round == gameweek

    # Match results
    results_train = dta.loc[train_idx, "FTR"].to_numpy()
    results_test = dta.loc[test_idx, "FTR"].to_numpy()

    # Column indices for odds
    cidx_home = [i for i, col in enumerate(dta.columns) if col.endswith("H") and col in feature_columns]
    cidx_draw = [i for i, col in enumerate(dta.columns) if col.endswith("D") and col in feature_columns]
    cidx_away = [i for i, col in enumerate(dta.columns) if col.endswith("A") and col in feature_columns]

    # Feature matrices for training
    feature_train_home = dta.loc[train_idx].iloc[:, cidx_home].to_numpy()
    feature_train_draw = dta.loc[train_idx].iloc[:, cidx_draw].to_numpy()
    feature_train_away = dta.loc[train_idx].iloc[:, cidx_away].to_numpy()

    # Feature matrices for testing
    feature_test_home = dta.loc[test_idx].iloc[:, cidx_home].to_numpy()
    feature_test_draw = dta.loc[test_idx].iloc[:, cidx_draw].to_numpy()
    feature_test_away = dta.loc[test_idx].iloc[:, cidx_away].to_numpy()

    train_arrays = [feature_train_home, feature_train_draw, feature_train_away]
    test_arrays = [feature_test_home, feature_test_draw, feature_test_away]

    imputed_training_matrices = []
    imputed_test_matrices = []

    for idx, farray in enumerate(train_arrays):
        imp = SimpleImputer(strategy="mean")
        farray = imp.fit_transform(farray)
        test_arrays[idx] = imp.transform(test_arrays[idx])

        imputed_training_matrices.append(farray)
        imputed_test_matrices.append(test_arrays[idx])

    # Merge the imputed arrays
    feature_train = np.concatenate(imputed_training_matrices, axis=1)
    feature_test = np.concatenate(imputed_test_matrices, axis=1)

    # Train AdaBoost
    adb = AdaBoostClassifier(
        DecisionTreeClassifier(max_depth=3),
        n_estimators=50000,
        learning_rate=0.2,
        random_state=42
    )
    adb = adb.fit(feature_train, results_train)

    training_pred = adb.predict(feature_train)
    print(skm.confusion_matrix(results_train, training_pred))

    test_pred = adb.predict(feature_test)

    gameweek_predictions = dta.loc[dta['Round'] == gameweek, historic_final_columns].copy()
    gameweek_predictions['PredictedResult'] = test_pred

    return gameweek_predictions


# ----------------------
# Evaluation pipeline
# ----------------------
models = [ada_model]

total_fixtures = 0
total_predicted = 0


def eval_predictions(gameweek_predictions):
    global total_fixtures
    global total_predicted

    no_predicted = 0
    if gameweek_predictions['FTR'].isnull().all():
        print("no full time results")
    else:
        print("evaluate results")
        for _, row in gameweek_predictions.iterrows():
            if row['FTR'] == row['PredictedResult']:
                no_predicted += 1

        total_fixtures += len(gameweek_predictions)
        total_predicted += no_predicted

    return no_predicted / len(gameweek_predictions)


def evaluate_latest_gw_run():
    predictions = None
    for m in models:
        weekly_score = []
        for i in range(2, 4):
            predictions = m("Downloads/E0.csv", i)
            n = eval_predictions(predictions)
            print(n)
            weekly_score.append(n)
        print(weekly_score)
        histogram_plot_score(weekly_score)
        z_score = sum(weekly_score) / len(weekly_score)
        print(z_score)

    return predictions


a = evaluate_latest_gw_run()


C:\Users\Mark\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
C:\Users\Mark\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


[[3 0 0]
 [0 3 0]
 [0 0 4]]
evaluate results
0.5
[[5 0 0]
 [0 7 0]
 [0 0 8]]
evaluate results
0.3
[0.5, 0.3]


NameError: name 'plt' is not defined

In [2]:
def histogram_plot_score(data):
    #print(data)
    n_bins = len(data)
    plt.hist(data)
    plt.hist(data,n_bins,histtype='bar')
    plt.show()
    return

In [ ]:
def evaluate_season_by_gw_run():    
    for m in models:
        weekly_score = []
        for i in range(2,30):
            #print("GW %s:" % i)
            predictions = m("E0.csv",i)
            #predictions = m("2017_2018.csv",i)
            #print (predictions)
            predictions['TakenOdds'] = collect_predicted_odds(predictions)
            n = eval_predictions(predictions)
            #print (n)
            # not always 10 events so normalized to no_correct / len(predictions)
            weekly_score.append(n)
        print("Season PredRate: %s" % weekly_score)
        #histogram_plot_score(weekly_score)
        z_score = sum(weekly_score)/len(weekly_score)
        print("Season Z-Score: %s" % z_score)
        nCombs = nCk(predictions,2)
        print("Number of bets %s:" % nCombs)
        correct_results = predictions[predictions['FTR'] == predictions['PredictedResult']]
        doubles_method(correct_results)
        
    return predictions

In [ ]:
b = evaluate_season_by_gw_run()

In [ ]:
def collect_predicted_odds(predictions):
    
    predicted_odds = []
                       
    
    for index,row in predictions.iterrows(): 
        if row['PredictedResult'] == 'H':
            predicted_odds.append(row['B365H'])
        elif row['PredictedResult'] == 'D':
            predicted_odds.append(row['B365D'])
        else:
            predicted_odds.append(row['B365A'])
    #odds = predictions['B365H']#,'B365D','B365A']
    #print (1./odds)
    
    return predicted_odds

In [ ]:
def calculate_simple_features(self, data):
    enhanced_data = data.copy().sort_values(['season']).reset_index(drop=True)
    
    enhanced_data['home_team_strength'] = 50
    enhanced_data['away_team_strength'] = 50
    enhanced_data['home_recent_form'] = 5
    enhanced_data['away_recent_form'] = 5

    enhanced_data['home_goals_avg'] = 1.5
    enhanced_data['away_goals_avg'] = 1.5

    enhanced_data['home_goals_conceded_avg'] = 1.5
    enhanced_data['away_goals_conceded_avg'] = 1.5

    enhanced_data['home_advantage'] = 1

    for i, match in enhanced_data.iterrows():
        home_team = match['home_team']
        away_team = match['away_team']

        home_history = self.get_team_history(enhanced_data, home_team, i, games=5)
        away_history = self.get_team_history(enhanced_data, away_team, i, games=5)

        home_stats = self.calculate_team_stats(home_history, home_team)
        away_stats = self.calculate_team_stats(away_history, home_team)

        enhanced_data.loc[i, 'home_team_strength'] = home_stats['strength']
        enhanced_data.loc[i, 'away_team_strength'] = away_stats['strength']
        
        enhanced_data.loc[i, 'away_recent_strength'] = away_stats['form']
        enhanced_data.loc[i, 'home_recent_strength'] = home_stats['form']
        
    
    
    

In [5]:
def walkforward_evaluation(filename):
    dta = pd.read_csv(filename)
    max_gw = dta.Round.max()

    total_fixtures = 0
    total_correct = 0
    all_scores = []

    for gw in range(2, max_gw + 1):  # start from 2 (need history)
        predictions = ada_model(filename, gw)

        # Evaluate
        correct = (predictions['FTR'] == predictions['PredictedResult']).sum()
        total = len(predictions)
        accuracy = correct / total if total > 0 else 0

        # Accumulate
        total_fixtures += total
        total_correct += correct
        all_scores.append(accuracy)

        # Print round results
        print(f"\n=== Gameweek {gw} ===")
        print(predictions[['HomeTeam', 'AwayTeam', 'FTR', 'PredictedResult']])
        print(f"Correct: {correct}/{total} ({accuracy:.1%})")

    # Overall performance
    overall_accuracy = total_correct / total_fixtures if total_fixtures > 0 else 0
    print("\n=======================")
    print(f"Walk-forward complete over {max_gw-1} weeks")
    print(f"Total Fixtures: {total_fixtures}")
    print(f"Total Correct : {total_correct}")
    print(f"Overall Accuracy: {overall_accuracy:.2%}")

    return all_scores


In [ ]:
c = walkforward_evaluation('Downloads/E0.csv')

C:\Users\Mark\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
C:\Users\Mark\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


[[3 0 0]
 [0 3 0]
 [0 0 4]]

=== Gameweek 2 ===
            HomeTeam        AwayTeam FTR PredictedResult
10           Arsenal         Burnley   H               H
11       Aston Villa     Bournemouth   A               H
12          Brighton        West Ham   D               H
13           Everton         Watford   H               A
14           Norwich       Newcastle   H               H
15       Southampton       Liverpool   A               A
16          Man City       Tottenham   D               H
17  Sheffield United  Crystal Palace   H               H
18           Chelsea       Leicester   D               A
19            Wolves      Man United   D               D
Correct: 5/10 (50.0%)
[[5 0 0]
 [0 7 0]
 [0 0 8]]

=== Gameweek 3 ===
            HomeTeam        AwayTeam FTR PredictedResult
20       Aston Villa         Everton   H               A
21           Norwich         Chelsea   A               A
22          Brighton     Southampton   A               H
23        Man United  Cryst

C:\Users\Mark\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


[[12  0  0]
 [ 0  8  0]
 [ 0  0 10]]

=== Gameweek 4 ===
          HomeTeam          AwayTeam FTR PredictedResult
30     Southampton        Man United   D               H
31         Chelsea  Sheffield United   D               A
32  Crystal Palace       Aston Villa   H               A
33       Leicester       Bournemouth   H               D
34        Man City          Brighton   H               H
35       Newcastle           Watford   D               H
36        West Ham           Norwich   H               D
37         Burnley         Liverpool   A               A
38         Everton            Wolves   H               D
39         Arsenal         Tottenham   D               A
Correct: 2/10 (20.0%)


C:\Users\Mark\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


...

C:\Users\Mark\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


[[61  0  0]
 [ 0 46  0]
 [ 0  0 82]]

=== Gameweek 20 ===
        HomeTeam          AwayTeam FTR PredictedResult
189     Brighton       Bournemouth   H               H
190    Newcastle           Everton   A               D
191  Southampton    Crystal Palace   D               H
192      Watford       Aston Villa   H               H
193      Norwich         Tottenham   D               A
194     West Ham         Leicester   A               A
195      Burnley        Man United   A               H
196      Arsenal           Chelsea   A               A
197    Liverpool            Wolves   H               H
198     Man City  Sheffield United   H               H
Correct: 6/10 (60.0%)


C:\Users\Mark\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


[[65  0  0]
 [ 0 48  0]
 [ 0  0 86]]

=== Gameweek 21 ===
        HomeTeam          AwayTeam FTR PredictedResult
199     Brighton           Chelsea   D               A
200      Burnley       Aston Villa   A               H
201    Newcastle         Leicester   A               A
202  Southampton         Tottenham   H               A
203      Watford            Wolves   H               D
204     Man City           Everton   H               A
205      Norwich    Crystal Palace   D               H
206     West Ham       Bournemouth   H               A
207      Arsenal        Man United   H               D
208    Liverpool  Sheffield United   H               H
Correct: 2/10 (20.0%)


C:\Users\Mark\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
